# RAG Based Profile Matching: experiments and analysis

This notebook looks at the matcher from a few angles:

1. What the dataset looks like
2. How resumes are chunked
3. One search, end to end
4. Semantic vs keyword vs hybrid search
5. What the must-have filter changes
6. Skill spellings (K8s vs Kubernetes)
7. The RRF constant k
8. How the 0-100 score is built
9. Speed

Run the cells from top to bottom. The first cell builds the database if it is missing.

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import resume_rag
import job_matcher
import evaluate

if not os.path.exists("data/ground_truth.json"):
    import generate_dataset
    generate_dataset.main()

if not os.path.exists("vector_db/candidates.json"):
    resume_rag.main()

resources = job_matcher.load_resources()
truth = evaluate.load_ground_truth()
jobs = evaluate.load_jobs()
print(len(resources["candidates"]), "candidates,", resources["collection"].count(), "chunks,", len(jobs), "job descriptions")

## 1. The dataset

36 made-up resumes across six kinds of role, with experience from 1 to 11 years.
The files are a mix of PDF, DOCX and TXT, and the section titles change from resume
to resume ("EXPERIENCE", "WORK EXPERIENCE", "PROFESSIONAL EXPERIENCE"), so the
reading and chunking code has to cope with that.

In [ ]:
family_by_path = {}
for record in truth["candidates"]:
    family_by_path[record["resume_path"]] = record["family"]

rows = []
for candidate in resources["candidates"]:
    rows.append({
        "name": candidate["candidate_name"],
        "file type": os.path.splitext(candidate["resume_path"])[1],
        "role family": family_by_path[candidate["resume_path"]],
        "years": candidate["experience_years"],
        "skills": len(candidate["skills"]),
        "education": candidate["education"],
    })
candidates_table = pd.DataFrame(rows)
candidates_table.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

candidates_table["file type"].value_counts().plot(kind="bar", ax=axes[0], title="Resumes by file type")
candidates_table["role family"].value_counts().plot(kind="bar", ax=axes[1], title="Resumes by role family")
candidates_table["years"].plot(kind="hist", bins=11, ax=axes[2], title="Years of experience")

plt.tight_layout()
plt.show()

## 2. Chunking by section

Each resume is split at its headings, so one chunk is one section (summary, skills,
experience, ...). A section longer than 800 characters would be cut between lines,
but in this dataset almost every section fits in one chunk.

Why not fixed-size chunks? A 500-character window could start in the middle of one
job and end in the middle of the next, and the skills line could end up glued to the
education line. Section chunks keep related lines together, and the section name
becomes useful metadata ("this match came from the experience section").

In [ ]:
example = resources["candidates"][0]
example_chunks = resources["collection"].get(where={"resume_path": example["resume_path"]})

for text, metadata in zip(example_chunks["documents"], example_chunks["metadatas"]):
    print("-----", metadata["section"], "-----")
    print(text)
    print()

In [ ]:
all_chunks = resources["collection"].get()
chunk_rows = []
for text, metadata in zip(all_chunks["documents"], all_chunks["metadatas"]):
    chunk_rows.append({"section": metadata["section"], "characters": len(text)})
chunk_table = pd.DataFrame(chunk_rows)

print("Chunks per section type:")
print(chunk_table["section"].value_counts().to_string())
print()
print("Chunk length: mean", round(chunk_table["characters"].mean()), "characters, longest", chunk_table["characters"].max())

## 3. One search, end to end

The Python backend job asks for Python and Django as must-haves and at least 5 years.

In [ ]:
jd_text = open("data/job_descriptions/jd_01_python_backend.txt").read()
output, info = job_matcher.match_job(jd_text)

print("Parsed job:", info["job"]["must_have"], "| nice to have:", info["job"]["nice_to_have"], "| min years:", info["job"]["min_years"])
print("Search took", round(info["seconds"] * 1000), "ms")

pd.DataFrame([{
    "candidate": m["candidate_name"],
    "score": m["match_score"],
    "matched skills": ", ".join(m["matched_skills"]),
} for m in output["top_matches"]])

In [ ]:
print(json.dumps(output["top_matches"][0], indent=2))

## 4. Semantic vs keyword vs hybrid

To compare the search methods fairly, the must-have filter is switched off here.
With the filter on, all three methods only see the 3 or 4 qualified people and
would all look perfect.

- **Precision@5**: how many of the top 5 do this kind of job (role family)
- **Recall@10**: how many of the qualified candidates are in the top 10
- **MRR**: 1 / the rank of the first strong match. 1.0 means the first result was a strong match.

In [ ]:
retrieval = evaluate.retrieval_quality(jobs, truth)

summary = pd.DataFrame({
    mode: {
        "Precision@5": retrieval[mode]["precision_at_5"],
        "Recall@10": retrieval[mode]["recall_at_10"],
        "MRR": retrieval[mode]["mrr"],
    }
    for mode in ["semantic", "keyword", "hybrid"]
}).T
print(summary)

summary.plot(kind="bar", figsize=(8, 4), ylim=(0, 1.05), title="Retrieval quality without the filter")
plt.xticks(rotation=0)
plt.show()

In [ ]:
per_job = []
for mode in ["semantic", "keyword", "hybrid"]:
    for row in retrieval[mode]["per_job"]:
        per_job.append({"job": row["job"], "method": mode, "MRR": row["mrr"]})
pd.DataFrame(per_job).pivot(index="job", columns="method", values="MRR")

In [ ]:
# The same numbers in words
best_mrr = max(["semantic", "keyword", "hybrid"], key=lambda mode: retrieval[mode]["mrr"])
print("Best MRR:", best_mrr, retrieval[best_mrr]["mrr"])
print("Hybrid vs semantic MRR:", retrieval["hybrid"]["mrr"], "vs", retrieval["semantic"]["mrr"])
print("Hybrid vs keyword MRR: ", retrieval["hybrid"]["mrr"], "vs", retrieval["keyword"]["mrr"])

## 5. What the must-have filter changes

Below are the top 10 for the Python job **without** the filter. The "qualified" column
shows who really has Python, Django and 5+ years. Without the filter, people with
the right skills but too little experience still rank high, because their resumes
read almost the same.

In [ ]:
qualified = set(truth["jobs"]["jd_01_python_backend"]["qualified"])
no_filter, _ = job_matcher.match_job(jd_text, use_filter=False)

rows = []
for match in no_filter["top_matches"]:
    candidate = next(c for c in resources["candidates"] if c["resume_path"] == match["resume_path"])
    rows.append({
        "candidate": match["candidate_name"],
        "score": match["match_score"],
        "years": candidate["experience_years"],
        "has Django": "Django" in candidate["skills"],
        "qualified": match["resume_path"] in qualified,
    })
pd.DataFrame(rows)

In [ ]:
filter_rows = evaluate.filter_quality(jobs, truth)
pd.DataFrame(filter_rows)

## 6. Skill spellings

Some resumes write "K8s" instead of "Kubernetes", "Postgres" instead of "PostgreSQL"
or "ML" instead of "Machine Learning". Both the resumes and the job descriptions are
mapped to one standard name, so these two searches should return the same people.
The order can move a little, because the embedding model still sees different words.

In [ ]:
short = job_matcher.match_job("DevOps engineer, 4+ years K8s, Terraform on AWS")[0]
long = job_matcher.match_job("DevOps engineer, 4+ years Kubernetes, Terraform on AWS")[0]

short_names = [m["candidate_name"] for m in short["top_matches"]]
long_names = [m["candidate_name"] for m in long["top_matches"]]
print("K8s:       ", short_names)
print("Kubernetes:", long_names)
print("Same people:", set(short_names) == set(long_names))

import re
alias_users = []
for candidate in resources["candidates"]:
    if re.search(r"\b(k8s|postgres|ml|js)\b", candidate["full_text"].lower()):
        alias_users.append(candidate["candidate_name"])
print("\nResumes that use a short spelling:", ", ".join(alias_users))

## 7. The RRF constant k

RRF gives each candidate 1 / (k + rank) from each list. A small k gives much more
weight to the first places, a big k makes the lists count more evenly.

In [ ]:
original_k = job_matcher.RRF_K
rows = []
for k in [1, 10, 60, 200]:
    job_matcher.RRF_K = k
    result = evaluate.retrieval_quality(jobs, truth)["hybrid"]
    rows.append({"k": k, "Precision@5": result["precision_at_5"], "Recall@10": result["recall_at_10"], "MRR": result["mrr"]})
job_matcher.RRF_K = original_k
pd.DataFrame(rows)

## 8. How the score is built

The score has three parts: meaning (45 points), skill coverage (35) and experience (20).
Seeing the parts makes it easy to explain why one person beat another.

In [ ]:
job = info["job"]
sections = job_matcher.semantic_search(job, use_filter=True)

rows = []
for match in output["top_matches"]:
    candidate = next(c for c in resources["candidates"] if c["resume_path"] == match["resume_path"])
    details = job_matcher.score_candidate(candidate, job, sections[match["resume_path"]])
    similarity = details["best_similarity"]
    semantic_points = job_matcher.SEMANTIC_POINTS * min(max((similarity - job_matcher.SIMILARITY_LOW) /
                      (job_matcher.SIMILARITY_HIGH - job_matcher.SIMILARITY_LOW), 0), 1)
    possible = 2 * len(job["must_have"]) + len(job["nice_to_have"])
    skill_points = job_matcher.SKILL_POINTS * (2 * len(details["matched_must"]) + len(details["matched_nice"])) / possible
    experience_points = job_matcher.EXPERIENCE_POINTS * min(candidate["experience_years"] / job["min_years"], 1.5) / 1.5
    rows.append({
        "candidate": match["candidate_name"],
        "meaning": round(semantic_points, 1),
        "skills": round(skill_points, 1),
        "experience": round(experience_points, 1),
        "total": details["score"],
    })

parts = pd.DataFrame(rows).set_index("candidate")
print(parts)
parts[["meaning", "skills", "experience"]].plot(kind="barh", stacked=True, figsize=(8, 3), title="Score parts")
plt.show()

## 9. Speed

In [ ]:
timings = []
job_matcher.match_job(jobs[0][1])       # warm-up
for job_id, text in jobs:
    for _ in range(10):
        start = time.time()
        job_matcher.match_job(text)
        timings.append((time.time() - start) * 1000)

print("Searches:", len(timings))
print("Mean:", round(np.mean(timings), 1), "ms | p95:", round(np.percentile(timings, 95), 1), "ms")

with open("output/ingestion_stats.json") as file:
    print("Ingestion:", json.load(file))

plt.hist(timings, bins=20)
plt.title("Search latency (ms)")
plt.show()

## Takeaways

- **The filter does the heavy lifting for hard requirements.** Embeddings cannot tell
  "4 years" from "6 years" and do not know that Django is required. Rules on clean
  metadata handle that reliably, and the vector search then only has to order the people who qualify.
- **Section chunks pay off.** Each excerpt in the output is a whole section, and the
  section name says where the match came from, which makes the reasoning easy to check.
- **Keyword search is strong when the job lists exact skills.** Mapping spellings to
  one standard name (K8s, Postgres, ML) is what makes it reliable.
- **Semantic search helps with wording.** It ranks people whose experience reads like
  the job description, even when the skill words differ.
- **Hybrid with RRF is the safer default.** Compare the MRR numbers in section 4: on
  this data it is usually at least as good as the weaker method and often close to the stronger one.
- **Limits.** The dataset is generated, so it is cleaner than real resumes. Real resumes
  have tables, columns, scanned pages and skills that only appear in experience lines,
  so extraction accuracy would drop and the skill list would need to grow.